# 04 — Locked final runs and RevisitOP evaluation

Only run this notebook after Notebook 03 has written `MyDrive/lightweight-cbir/locked/locked_run.json` from SfM validation alone. Revisited Oxford and Paris are held out: their scores must never change the selected checkpoint, layers, temperatures, or training settings.

In [ ]:
# Run this first in every fresh Colab runtime.  It intentionally does not use PYTHONPATH.
from pathlib import Path
import importlib
import os
import shutil
import subprocess
import sys

# Override this only if you use a fork/private clone URL.
REPO_URL = os.environ.get('CBIR_REPO_URL', 'https://github.com/armin-faraji/LightweightCBIR.git')
REPO_REVISION = os.environ.get('CBIR_REPO_REVISION', 'main')
PROJECT_ROOT = Path('/content/lightweight-cbir')
if not PROJECT_ROOT.is_dir():
    subprocess.run(['git', 'clone', REPO_URL, str(PROJECT_ROOT)], check=True)
    subprocess.run(['git', '-C', str(PROJECT_ROOT), 'checkout', REPO_REVISION], check=True)
os.chdir(PROJECT_ROOT)
required_project_files = (PROJECT_ROOT / 'pyproject.toml', PROJECT_ROOT / 'src' / 'cbir' / '__init__.py', PROJECT_ROOT / 'src' / 'cbir' / 'artifacts.py')
missing_project_files = [str(path.relative_to(PROJECT_ROOT)) for path in required_project_files if not path.is_file()]
if missing_project_files:
    raise RuntimeError('The cloned repository does not contain the Colab-ready project code: ' + ', '.join(missing_project_files) + '. Push the current repository, or set CBIR_REPO_URL/CBIR_REPO_REVISION to the matching commit.')
# A regular reinstall makes the cloned package visible in this current kernel.
subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet', '--no-deps', '--force-reinstall', '.'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet', 'h5py', 'scipy', 'PyYAML', 'tqdm', 'matplotlib', 'Pillow'], check=True)

importlib.invalidate_caches()
try:
    import cbir
except ModuleNotFoundError as error:
    raise RuntimeError('The package installation completed but cbir is not visible to this kernel. Restart the runtime once, then rerun this first cell.') from error
print('cbir package:', cbir.__file__)

from cbir.artifacts import create_artifact_run, make_artifact_run_id
from cbir.cloud import mount_colab_drive, runtime_report, write_runtime_report
from cbir.config import config_to_dict, load_project_config
from cbir.utils import stable_hash

PERSISTENT_ROOT = mount_colab_drive() / 'lightweight-cbir'
PERSISTENT_ROOT.mkdir(parents=True, exist_ok=True)
os.environ.setdefault('TORCH_HOME', str(PERSISTENT_ROOT / 'torch_hub'))
CONFIG_PATH = Path('configs/colab.yaml')
cfg = load_project_config(CONFIG_PATH)
environment = runtime_report(project_root=PROJECT_ROOT)
config_fingerprint = stable_hash(config_to_dict(cfg))
RUN_ID = make_artifact_run_id(notebook='04', git_sha=environment['git_sha'], config_fingerprint=config_fingerprint)
ARTIFACT_RUN = create_artifact_run(PROJECT_ROOT / 'outputs', '04', run_id=RUN_ID, metadata={'git_sha': environment['git_sha'], 'config_fingerprint': config_fingerprint})
LOCAL_OUTPUT_DIR = ARTIFACT_RUN.local_dir
DRIVE_OUTPUT_ROOT = PERSISTENT_ROOT / 'notebook_outputs'
write_runtime_report(LOCAL_OUTPUT_DIR, project_root=PROJECT_ROOT, extra={'notebook': '04', 'config': str(CONFIG_PATH)})
shutil.copy2(CONFIG_PATH, ARTIFACT_RUN.path_for('config.yaml'))
print('Project:', PROJECT_ROOT)
print('Artifacts:', LOCAL_OUTPUT_DIR)

## Restore the immutable SfM-selected checkpoint

The lock contains the Drive path of a checkpoint that was already published by Notebook 03. The checkpoint is copied to fast local storage before evaluation; RevisitOP never reads a large model file through Drive/FUSE.

In [ ]:
from cbir.cache import sha256_file
from cbir.cloud import stage_file
from cbir.utils import read_json

LOCK_PATH = PERSISTENT_ROOT / 'locked' / 'locked_run.json'
if not LOCK_PATH.is_file():
    raise FileNotFoundError(f'Notebook 03 has not published a locked checkpoint: {LOCK_PATH}')
locked_run = read_json(LOCK_PATH)
if locked_run.get('selected_on') != 'SfM-30k validation only':
    raise ValueError('Refusing a checkpoint whose lock does not state SfM-only selection.')
DRIVE_CHECKPOINT = Path(locked_run['checkpoint_path'])
if not DRIVE_CHECKPOINT.is_relative_to(PERSISTENT_ROOT):
    raise ValueError(f'Locked checkpoint is outside the project Drive root: {DRIVE_CHECKPOINT}')
LOCKED_CHECKPOINT = ARTIFACT_RUN.path_for('locked_checkpoint/best.pt')
stage_file(DRIVE_CHECKPOINT, LOCKED_CHECKPOINT)
if sha256_file(LOCKED_CHECKPOINT) != locked_run.get('checkpoint_sha256'):
    raise RuntimeError('Staged checkpoint checksum does not match the SfM lock.')
ARTIFACT_RUN.write_json('locked_run.json', locked_run)
print('Using locked checkpoint:', LOCKED_CHECKPOINT)

## Stage Revisited Oxford and Paris

On the first run this downloads and validates the official archives into `/content`, then publishes a validated copy to Drive. On later runtimes it stages the Drive copy back to fast storage. Full JPEG decoding is deliberate: it catches broken downloads before a final benchmark run.

In [ ]:
LOCAL_REVISIT_ROOT = Path('/content/cbir_data/revisitop')
DRIVE_REVISIT_ROOT = PERSISTENT_ROOT / 'datasets' / 'revisitop'
DRIVE_REVISIT_ROOT.mkdir(parents=True, exist_ok=True)
# auto stages a valid Drive dataset when present and downloads only a missing one.
# --publish-root then validates and atomically publishes any new local copy.
stage_command = [
    sys.executable, 'scripts/prepare_revisitop.py',
    '--output-root', str(LOCAL_REVISIT_ROOT),
    '--source-root', str(DRIVE_REVISIT_ROOT),
    '--mode', 'auto',
    '--publish-root', str(DRIVE_REVISIT_ROOT),
    '--repair',
]
subprocess.run(stage_command, cwd=PROJECT_ROOT, check=True)
ARTIFACT_RUN.write_json(
    'revisitop_stage.json',
    read_json(LOCAL_REVISIT_ROOT / 'revisitop_preparation_report.json'),
)
print('RevisitOP staged locally at', LOCAL_REVISIT_ROOT)

## Run the official Medium and Hard protocols

The evaluator extracts full-image database descriptors and one descriptor from each official query bounding-box crop. It caches descriptor bundles under this run directory, so rerunning the cell reuses matching bundles rather than repeating backbone inference.

In [ ]:
from cbir.plotting import SeriesData, plot_series
from cbir.utils import read_json

BACKBONE_BATCH_SIZE = 8
CHUNK_SIZE = 128
LOCAL_EVALUATION_ROOT = LOCAL_OUTPUT_DIR / 'evaluation'
LOCAL_EVALUATION_ROOT.mkdir(parents=True, exist_ok=True)
evaluation_reports = {}
for dataset_name in ('roxford5k', 'rparis6k'):
    command = [
        sys.executable, 'scripts/evaluate_revisitop.py',
        '--config', str(CONFIG_PATH),
        '--checkpoint', str(LOCKED_CHECKPOINT),
        '--revisit-root', str(LOCAL_REVISIT_ROOT),
        '--dataset', dataset_name,
        '--output-dir', str(LOCAL_EVALUATION_ROOT),
        '--backbone-batch-size', str(BACKBONE_BATCH_SIZE),
        '--chunk-size', str(CHUNK_SIZE),
    ]
    subprocess.run(command, cwd=PROJECT_ROOT, check=True)
    evaluation_reports[dataset_name] = read_json(
        LOCAL_EVALUATION_ROOT / dataset_name / 'evaluation_report.json'
    )

map_series = {
    'ROxford5k': SeriesData(
        x=[0, 1],
        y=[evaluation_reports['roxford5k']['medium']['map'], evaluation_reports['roxford5k']['hard']['map']],
    ),
    'RParis6k': SeriesData(
        x=[0, 1],
        y=[evaluation_reports['rparis6k']['medium']['map'], evaluation_reports['rparis6k']['hard']['map']],
    ),
}
figure, axis = plot_series(
    map_series,
    title='Locked-model RevisitOP mAP',
    xlabel='Protocol',
    ylabel='mAP',
)
axis.set_xticks([0, 1], ['Medium', 'Hard'])
ARTIFACT_RUN.save_figure('figures/revisitop_map_medium_hard.png', figure)
ARTIFACT_RUN.write_json('revisitop_summary.json', {
    'locked_checkpoint': locked_run,
    'backbone_batch_size': BACKBONE_BATCH_SIZE,
    'chunk_size': CHUNK_SIZE,
    'reports': evaluation_reports,
})
print({name: {'medium_map': report['medium']['map'], 'hard_map': report['hard']['map']} for name, report in evaluation_reports.items()})
figure

## Publish notebook outputs to Drive

Run this as the final cell. It validates and publishes the plots, per-query evaluation reports, descriptor-bundle provenance, lock copy, configuration, and runtime details.

In [ ]:
published_output = ARTIFACT_RUN.publish(DRIVE_OUTPUT_ROOT)
print('Validated notebook artifacts published to:', published_output)